# L4 — Sensitive Data Masking

## What you'll learn

- How to mask customer data in tool responses before the orchestrator sees it
- How to create a Bedrock Guardrail that anonymizes personal data and blocks prompt injection
- How to attach an interceptor to the Gateway so every tool response is checked
- How to verify that email addresses and shipping addresses are masked

## Overview

Tool responses can contain customer personal data, such as email addresses, shipping addresses and phone numbers. This notebook adds an **interceptor** to the Gateway that masks this data with a **Bedrock Guardrail** before the orchestrator sees it. The workshop has already set up the interceptor's permissions.

> **Compliance note:** in production, make sure your handling of personal, health and payment data meets the regulations that apply to you, such as GDPR, HIPAA and PCI-DSS. See [AWS compliance](https://aws.amazon.com/compliance/).

## Architecture

![Security layer: a tool call goes through the AgentCore Gateway to the order or refund tools; the raw response passes through the PII interceptor Lambda, which has the Bedrock Guardrail mask personal data before the masked response returns to the agents.](images/l4-sensitive-data-masking-architecture.png)

This notebook adds the PII interceptor and the guardrail in the middle of the diagram. The Gateway and the tools came from L3.

## Steps

1. Load the workshop configuration
2. Create the Bedrock Guardrail
3. Deploy the interceptor
4. Attach the interceptor to the Gateway
5. Test the masking

## License details

Refer to LICENSE file in the main folder for license details.

## Prerequisites

- [`1_gateway_tools.ipynb`](../l3-orchestration/1_gateway_tools.ipynb) is complete: the Gateway and the Order tool exist.
- You can run this notebook in parallel with [`3_orchestrator_agent.ipynb`](../l3-orchestration/3_orchestrator_agent.ipynb).

## Step 1: Load the workshop configuration

Install the AWS SDK, import the libraries, and read the Gateway, Cognito and interceptor role settings that the workshop saved earlier.

In [ ]:
%pip install --quiet --upgrade boto3
import boto3, json, time, io, zipfile, os
from botocore.exceptions import ClientError

REGION = os.environ.get("AWS_DEFAULT_REGION", "us-east-1")
SSM_PREFIX = "/anycompany/agentcore"

sts = boto3.client("sts", region_name=REGION)
ACCOUNT_ID = sts.get_caller_identity()["Account"]

ssm = boto3.client("ssm", region_name=REGION)
lambda_client = boto3.client("lambda", region_name=REGION)
bedrock = boto3.client("bedrock", region_name=REGION)
agentcore_control = boto3.client("bedrock-agentcore-control", region_name=REGION)

def get_ssm(name, default=None):
    try:
        return ssm.get_parameter(Name=name, WithDecryption=True)["Parameter"]["Value"]
    except ssm.exceptions.ParameterNotFound:
        if default is not None:
            return default
        raise

GATEWAY_ID = get_ssm(f"{SSM_PREFIX}/gateway_id")
GATEWAY_ARN = get_ssm(f"{SSM_PREFIX}/gateway_arn")
COGNITO_USER_POOL_ID = get_ssm(f"{SSM_PREFIX}/cognito_user_pool_id")
COGNITO_CLIENT_ID = get_ssm(f"{SSM_PREFIX}/cognito_client_id")
ORDERS_TABLE = get_ssm(f"{SSM_PREFIX}/dynamodb_orders_table")
# The interceptor Lambda's role is pre-provisioned by the L4 security stack.
INTERCEPTOR_ROLE_ARN = get_ssm(f"{SSM_PREFIX}/interceptor_role_arn")

print(f"Region:       {REGION}")
print(f"Gateway ID:   {GATEWAY_ID}")
print(f"Gateway ARN:  {GATEWAY_ARN}")
print(f"Cognito Pool: {COGNITO_USER_POOL_ID}")
print(f"Orders Table: {ORDERS_TABLE}")
print(f"Interceptor role: {INTERCEPTOR_ROLE_ARN}")

## Step 2: Create the Bedrock Guardrail

The interceptor applies this guardrail to every tool response. It:

- **Anonymizes personal data and credentials:** email, phone, name, address, credit cards, SSNs, passwords and AWS keys.
- **Blocks prompt-injection attempts** in input.

Create the guardrail (or reuse it if it already exists), publish a version, wait until it's ready, and save its ID and version for later.

In [ ]:
# Create (or reuse) the Bedrock Guardrail that anonymizes PII and blocks prompt injection,
# then publish its ID/version to SSM. The interceptor Lambda (Step 3) applies it to tool responses.
GUARDRAIL_NAME = "anycompany-pii-guardrail"

existing = next(
    (g for page in bedrock.get_paginator("list_guardrails").paginate()
     for g in page.get("guardrails", []) if g.get("name") == GUARDRAIL_NAME),
    None,
)
if existing:
    GUARDRAIL_ID = existing["id"]
    # Reuse the latest published version (fall back to publishing one if only DRAFT exists)
    versions = [g["version"] for page in bedrock.get_paginator("list_guardrails").paginate(guardrailIdentifier=GUARDRAIL_ID)
                for g in page.get("guardrails", []) if g.get("version") != "DRAFT"]
    GUARDRAIL_VERSION = max(versions, key=int) if versions else None
    print(f"Reusing existing guardrail {GUARDRAIL_NAME} ({GUARDRAIL_ID})")
else:
    _gr = bedrock.create_guardrail(
        name=GUARDRAIL_NAME,
        description="AnyCompany workshop guardrail: PII anonymization for Gateway tool responses",
        blockedInputMessaging="I can't process that request — it contains content that violates our safety or privacy policy.",
        blockedOutputsMessaging="I can't share that response — it would include content that violates our safety or privacy policy.",
        contentPolicyConfig={
            "filtersConfig": [
                {"type": "PROMPT_ATTACK", "inputStrength": "HIGH", "outputStrength": "NONE",
                 "inputAction": "BLOCK", "inputEnabled": True, "outputEnabled": False},
            ]
        },
        sensitiveInformationPolicyConfig={
            "piiEntitiesConfig": [
                # Credentials: anonymized so raw values never pass through.
                {"type": "CREDIT_DEBIT_CARD_NUMBER", "action": "ANONYMIZE"},
                {"type": "US_SOCIAL_SECURITY_NUMBER", "action": "ANONYMIZE"},
                {"type": "PASSWORD", "action": "ANONYMIZE"},
                {"type": "AWS_ACCESS_KEY", "action": "ANONYMIZE"},
                {"type": "AWS_SECRET_KEY", "action": "ANONYMIZE"},
                # Identifying PII: replaced with {EMAIL}, {PHONE}, {NAME}, {ADDRESS}.
                {"type": "EMAIL", "action": "ANONYMIZE"},
                {"type": "PHONE", "action": "ANONYMIZE"},
                {"type": "NAME", "action": "ANONYMIZE"},
                {"type": "ADDRESS", "action": "ANONYMIZE"},
            ]
        },
    )
    GUARDRAIL_ID = _gr["guardrailId"]
    GUARDRAIL_VERSION = None
    print(f"Created guardrail {GUARDRAIL_NAME} ({GUARDRAIL_ID})")

# Publish a stable version: the interceptor always references a published version,
# never DRAFT (which changes silently on config updates).
if not GUARDRAIL_VERSION:
    GUARDRAIL_VERSION = bedrock.create_guardrail_version(
        guardrailIdentifier=GUARDRAIL_ID, description="v1 — initial workshop config",
    )["version"]

# Wait for READY: Bedrock rejects apply calls while a guardrail is still CREATING (usually 1-5 s).
for _ in range(30):
    _status = bedrock.get_guardrail(guardrailIdentifier=GUARDRAIL_ID, guardrailVersion=GUARDRAIL_VERSION).get("status", "")
    if _status == "READY":
        break
    if _status == "FAILED":
        raise RuntimeError(f"Guardrail {GUARDRAIL_ID} failed to stabilize")
    time.sleep(2)
else:
    raise TimeoutError(f"Guardrail did not reach READY within 60s (last status: {_status})")

print(f"Guardrail ID:      {GUARDRAIL_ID}")
print(f"Guardrail Version: {GUARDRAIL_VERSION}")

# Publish to SSM for the interceptor deployment and for later notebooks
for key, value in {"guardrail_id": GUARDRAIL_ID, "guardrail_version": GUARDRAIL_VERSION}.items():
    ssm.put_parameter(Name=f"{SSM_PREFIX}/{key}", Value=str(value), Type="String", Overwrite=True)
    print(f"Published SSM: {SSM_PREFIX}/{key} = {value}")

## Step 3: Deploy the interceptor

Deploy the interceptor, a Lambda function named `anycompany-pii-interceptor`. For every tool response, it applies the guardrail to the response body before returning it to the orchestrator.

The Gateway needs two permissions to call it. The workshop has already granted the first, on the Gateway's role. The cell below adds the second, on the Lambda function, limited to this Gateway and this account.

In [ ]:
INTERCEPTOR_CODE = """import json, os, boto3, logging

logger = logging.getLogger()
logger.setLevel(logging.INFO)

bedrock_runtime = boto3.client('bedrock-runtime')

GUARDRAIL_ID = os.environ.get('GUARDRAIL_ID')
GUARDRAIL_VERSION = os.environ.get('GUARDRAIL_VERSION', 'DRAFT')


def mask_pii_with_guardrails(text):
    if not GUARDRAIL_ID:
        return text
    try:
        response = bedrock_runtime.apply_guardrail(
            guardrailIdentifier=GUARDRAIL_ID,
            guardrailVersion=GUARDRAIL_VERSION,
            source='OUTPUT',
            outputScope='FULL',
            content=[{'text': {'text': text}}]
        )
        outputs = response.get('outputs', [])
        if outputs:
            return outputs[0].get('text', text)
        return text
    except Exception as e:
        logger.error(f'Guardrail error: {e}')
        return text


def mask_tool_response(response_body):
    masked = json.loads(json.dumps(response_body))
    if 'result' not in masked or 'content' not in masked.get('result', {}):
        return masked
    for i, item in enumerate(masked['result']['content']):
        if item.get('type') != 'text' or not item.get('text'):
            continue
        text_val = item['text']
        try:
            parsed = json.loads(text_val)
            anonymized_str = mask_pii_with_guardrails(json.dumps(parsed, indent=2))
            # Always keep text as a string — MCP protocol requires content[i].text to be a string
            masked['result']['content'][i]['text'] = anonymized_str
        except (json.JSONDecodeError, TypeError):
            masked['result']['content'][i]['text'] = mask_pii_with_guardrails(text_val)
    return masked


def lambda_handler(event, context):
    logger.info(f'Interceptor invoked - method: {event.get("mcp", {}).get("gatewayRequest", {}).get("body", {}).get("method")}, status: {event.get("mcp", {}).get("gatewayResponse", {}).get("statusCode")}')
    try:
        mcp_data = event.get('mcp', {})
        gateway_response = mcp_data.get('gatewayResponse', {})
        gateway_request = mcp_data.get('gatewayRequest', {})
        response_headers = gateway_response.get('headers', {})
        response_body = gateway_response.get('body', {})
        status_code = gateway_response.get('statusCode', 200)
        method = gateway_request.get('body', {}).get('method', '')

        if method == 'tools/call':
            masked_body = mask_tool_response(response_body)
        else:
            masked_body = response_body

        return {
            'interceptorOutputVersion': '1.0',
            'mcp': {
                'transformedGatewayResponse': {
                    'headers': response_headers,
                    'body': masked_body,
                    'statusCode': status_code
                }
            }
        }
    except Exception as e:
        logger.error(f'Lambda error: {type(e).__name__}')
        gw = event.get('mcp', {}).get('gatewayResponse', {})
        return {
            'interceptorOutputVersion': '1.0',
            'mcp': {
                'transformedGatewayResponse': {
                    'headers': gw.get('headers', {}),
                    'body': gw.get('body', {}),
                    'statusCode': gw.get('statusCode', 500)
                }
            }
        }
"""

buf = io.BytesIO()
with zipfile.ZipFile(buf, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.writestr("lambda_function.py", INTERCEPTOR_CODE)
zip_bytes = buf.getvalue()

# NOTE: keep this name in sync with infra/stacks/l3-orchestration.yaml (gateway
# invoke grant) and infra/stacks/l4-security.yaml (role log-group scope).
INTERCEPTOR_LAMBDA_NAME = "anycompany-pii-interceptor"

try:
    resp = lambda_client.create_function(
        FunctionName=INTERCEPTOR_LAMBDA_NAME,
        Runtime="python3.12",
        Role=INTERCEPTOR_ROLE_ARN,
        Handler="lambda_function.lambda_handler",
        Code={"ZipFile": zip_bytes},
        Timeout=30,
        MemorySize=256,
        Environment={"Variables": {
            "GUARDRAIL_ID": GUARDRAIL_ID,
            "GUARDRAIL_VERSION": GUARDRAIL_VERSION,
        }},
    )
    INTERCEPTOR_LAMBDA_ARN = resp["FunctionArn"]
    print(f"Created: {INTERCEPTOR_LAMBDA_ARN}")
except lambda_client.exceptions.ResourceConflictException:
    INTERCEPTOR_LAMBDA_ARN = lambda_client.get_function(FunctionName=INTERCEPTOR_LAMBDA_NAME)["Configuration"]["FunctionArn"]
    lambda_client.update_function_code(FunctionName=INTERCEPTOR_LAMBDA_NAME, ZipFile=zip_bytes)
    # Wait for code update to complete before updating configuration
    lambda_client.get_waiter('function_updated_v2').wait(FunctionName=INTERCEPTOR_LAMBDA_NAME)
    lambda_client.update_function_configuration(
        FunctionName=INTERCEPTOR_LAMBDA_NAME,
        Role=INTERCEPTOR_ROLE_ARN,
        Environment={"Variables": {
            "GUARDRAIL_ID": GUARDRAIL_ID,
            "GUARDRAIL_VERSION": GUARDRAIL_VERSION,
        }},
    )
    print(f"Updated: {INTERCEPTOR_LAMBDA_ARN}")

# Allow the Gateway service to invoke the interceptor, scoped to THIS gateway + account
# (confused-deputy protection). Re-run safe: remove any prior statement first.
try:
    lambda_client.remove_permission(
        FunctionName=INTERCEPTOR_LAMBDA_NAME, StatementId="AllowGatewayInterceptorInvoke")
except lambda_client.exceptions.ResourceNotFoundException:
    pass
lambda_client.add_permission(
    FunctionName=INTERCEPTOR_LAMBDA_NAME,
    StatementId="AllowGatewayInterceptorInvoke",
    Action="lambda:InvokeFunction",
    Principal="bedrock-agentcore.amazonaws.com",
    SourceArn=GATEWAY_ARN,
    SourceAccount=ACCOUNT_ID,
)
print("Gateway invoke permission added (scoped to this gateway + account).")

# Wait for Lambda to be active
for _ in range(20):
    fn = lambda_client.get_function_configuration(FunctionName=INTERCEPTOR_LAMBDA_NAME)
    if fn.get("State") == "Active" and fn.get("LastUpdateStatus", "Successful") in ("Successful", None):
        print("Lambda Active and ready.")
        break
    time.sleep(5)

## Step 4: Attach the interceptor to the Gateway

Update the Gateway so that it runs the interceptor on every tool response, then wait until the Gateway is ready.

In [ ]:
# Get current gateway config to preserve existing settings
gw = agentcore_control.get_gateway(gatewayIdentifier=GATEWAY_ID)
print(f"Gateway status: {gw.get('status')}")
print(f"Current interceptors: {len(gw.get('interceptorConfigurations', []))}")

GATEWAY_ROLE_ARN = get_ssm(f"{SSM_PREFIX}/gateway_role_arn")
GATEWAY_NAME = get_ssm(f"{SSM_PREFIX}/gateway_name")
COGNITO_USER_POOL_ID = get_ssm(f"{SSM_PREFIX}/cognito_user_pool_id")
COGNITO_CLIENT_ID = get_ssm(f"{SSM_PREFIX}/cognito_client_id")

# Update gateway to add the interceptor — must include all required fields
try:
    agentcore_control.update_gateway(
        gatewayIdentifier=GATEWAY_ID,
        name=GATEWAY_NAME,
        roleArn=GATEWAY_ROLE_ARN,
        authorizerType="CUSTOM_JWT",
        authorizerConfiguration={
            "customJWTAuthorizer": {
                "discoveryUrl": f"https://cognito-idp.{REGION}.amazonaws.com/{COGNITO_USER_POOL_ID}/.well-known/openid-configuration",
                "allowedAudience": [COGNITO_CLIENT_ID],
            }
        },
        interceptorConfigurations=[
            {
                "interceptor": {
                    "lambda": {
                        "arn": INTERCEPTOR_LAMBDA_ARN
                    }
                },
                "interceptionPoints": ["RESPONSE"],
                "inputConfiguration": {
                    "passRequestHeaders": True
                }
            }
        ],
    )
    print("Gateway updated with PII interceptor.")
except Exception as e:
    print(f"Error updating gateway: {e}")
    raise

# Wait for gateway to be READY
print("Waiting for gateway to be ready...")
for _ in range(30):
    g = agentcore_control.get_gateway(gatewayIdentifier=GATEWAY_ID)
    status = g.get("status")
    if status == "READY":
        print("Gateway READY with interceptor attached.")
        break
    print(f"  {status}...")
    time.sleep(5)

## Step 5: Test the masking

Sign in as `gold_customer`, call the order tool for ORD-10001 through the Gateway, and check that the email and shipping address come back anonymized.

In [ ]:
import boto3, json
from botocore.auth import SigV4Auth
from botocore.awsrequest import AWSRequest
import requests as req_lib

# Retrieve the workshop test password from SSM (stored as SecureString).
# get_ssm() passes WithDecryption=True so we get the plaintext back.
USER_PASSWORD = get_ssm(f"{SSM_PREFIX}/user_password")

# Get a Cognito token for testing
# ⚠️ WORKSHOP ONLY — disposable Cognito test user; password fetched from SSM.
# In production, use federated identity (SAML/OIDC) instead of shared passwords.
cognito = boto3.client("cognito-idp", region_name=REGION)
auth_resp = cognito.initiate_auth(
    ClientId=COGNITO_CLIENT_ID,
    AuthFlow="USER_PASSWORD_AUTH",
    AuthParameters={"USERNAME": "gold_customer", "PASSWORD": USER_PASSWORD},
)
test_token = auth_resp["AuthenticationResult"]["IdToken"]
print("Got test token for gold_customer")

GATEWAY_URL = get_ssm(f"{SSM_PREFIX}/gateway_url")

# Find the order-tools target
targets = agentcore_control.list_gateway_targets(gatewayIdentifier=GATEWAY_ID)
order_target = None
for t in targets.get("targets", targets.get("items", [])):
    if "order" in t.get("name", "").lower():
        order_target = t
        break

if not order_target:
    print("Order tools target not found. Make sure 1_gateway_tools.ipynb has been run.")
else:
    print(f"Found order target: {order_target['name']}")

    # Call check_order_details via Gateway MCP
    mcp_request = {
        "jsonrpc": "2.0",
        "method": "tools/call",
        "id": 1,
        "params": {
            "name": f"{order_target['name']}___check_order_details",
            "arguments": {"order_id": "ORD-10001"}
        }
    }

    response = req_lib.post(
        GATEWAY_URL,
        headers={
            "Authorization": f"Bearer {test_token}",
            "Content-Type": "application/json",
        },
        json=mcp_request,
        timeout=30,
    )

    print(f"Response status: {response.status_code}")
    if response.status_code == 200:
        result = response.json()
        print(json.dumps(result, indent=2))
        print()
        print("Expected: contactInfo (email) and shippingAddress should be anonymized ({EMAIL}, {ADDRESS})")
    else:
        print(f"Error: {response.text}")

## Summary

This notebook added PII protection: a Bedrock Guardrail and an interceptor on the Gateway that anonymizes email addresses, addresses, phone numbers, card numbers and SSNs in every tool response before it reaches the orchestrator. The guardrail is saved for later use.

**Next:** [`1_decision_models.ipynb`](../l5-application/1_decision_models.ipynb). Add a decision step that triages each customer message.

---
## Cleanup

Deletes what this notebook created: the interceptor setting on the Gateway, the interceptor Lambda function and its log group, the guardrail, and its saved ID and version. The interceptor's IAM role and the Gateway's permission to call it belong to the workshop stack and are removed when it's deleted. The Gateway itself stays, because `1_gateway_tools.ipynb` owns it.

Run this **before** the cleanup in [`1_gateway_tools.ipynb`](../l3-orchestration/1_gateway_tools.ipynb), so the interceptor is detached from the Gateway before the Gateway is deleted. Uncomment the cell to run it.

In [ ]:
## === CLEANUP: Delete all resources created in this notebook ===
## ⚠️ Uncomment and run this cell only after you are done with ALL notebooks.
## NOTE: Run this BEFORE the L3 #1 (1_gateway_tools.ipynb) cleanup so the
##       interceptor is detached from the Gateway before L3 deletes the Gateway.

# import boto3, json, os
#
# REGION = os.environ.get("AWS_DEFAULT_REGION", "us-east-1")
# SSM_PREFIX = "/anycompany/agentcore"
#
# INTERCEPTOR_LAMBDA_NAME = "anycompany-pii-interceptor"
# INTERCEPTOR_LOG_GROUP   = f"/aws/lambda/{INTERCEPTOR_LAMBDA_NAME}"
# GUARDRAIL_NAME          = "anycompany-pii-guardrail"
#
# ssm               = boto3.client("ssm", region_name=REGION)
# lambda_client     = boto3.client("lambda", region_name=REGION)
# logs_client       = boto3.client("logs", region_name=REGION)
# agentcore_control = boto3.client("bedrock-agentcore-control", region_name=REGION)
# bedrock           = boto3.client("bedrock", region_name=REGION)
# sts               = boto3.client("sts", region_name=REGION)
#
# ACCOUNT_ID = sts.get_caller_identity()["Account"]
# INTERCEPTOR_LAMBDA_ARN = f"arn:aws:lambda:{REGION}:{ACCOUNT_ID}:function:{INTERCEPTOR_LAMBDA_NAME}"
#
# def _ssm_get(key):
#     try:
#         return ssm.get_parameter(Name=f"{SSM_PREFIX}/{key}")["Parameter"]["Value"]
#     except ssm.exceptions.ParameterNotFound:
#         return None
#
# # ── Phase 1: Remove interceptor from Gateway ────────────────────────────────
# # NOTE: update_gateway does not accept empty interceptorConfigurations=[].
# # Omit the parameter entirely to remove all interceptors.
# gateway_id   = _ssm_get("gateway_id")
# gateway_name = _ssm_get("gateway_name")
# gateway_role = _ssm_get("gateway_role_arn")
# cog_pool     = _ssm_get("cognito_user_pool_id")
# cog_client   = _ssm_get("cognito_client_id")
#
# if all([gateway_id, gateway_name, gateway_role, cog_pool, cog_client]):
#     try:
#         agentcore_control.update_gateway(
#             gatewayIdentifier=gateway_id,
#             name=gateway_name,
#             roleArn=gateway_role,
#             authorizerType="CUSTOM_JWT",
#             authorizerConfiguration={
#                 "customJWTAuthorizer": {
#                     "discoveryUrl": f"https://cognito-idp.{REGION}.amazonaws.com/{cog_pool}/.well-known/openid-configuration",
#                     "allowedAudience": [cog_client],
#                 }
#             },
#         )
#         print("Removed interceptor from Gateway.")
#     except agentcore_control.exceptions.ResourceNotFoundException:
#         print("Gateway already deleted; skipping interceptor detach.")
#     except Exception as e:
#         print(f"Update gateway: {e}")
# else:
#     print("Gateway/Cognito SSM keys missing; skipping interceptor detach (L3 #1 likely cleaned up first).")
#
# # ── Phase 2: Gateway invoke-permission is managed by CloudFormation ─────────
# # The interceptor is in the Gateway role's InvokeLambdaTargets (l3-orchestration.yaml)
# # and removed with the stack. Nothing to revert here.
# print("Gateway invoke-permission is CloudFormation-managed; nothing to revert.")
#
# # ── Phase 3: Delete interceptor Lambda ──────────────────────────────────────
# try:
#     lambda_client.delete_function(FunctionName=INTERCEPTOR_LAMBDA_NAME)
#     print(f"Deleted Lambda: {INTERCEPTOR_LAMBDA_NAME}")
# except lambda_client.exceptions.ResourceNotFoundException:
#     print(f"Lambda {INTERCEPTOR_LAMBDA_NAME} already deleted.")
# except Exception as e:
#     print(f"Lambda delete: {e}")
#
# # ── Phase 4: Delete Lambda CloudWatch log group ─────────────────────────────
# try:
#     logs_client.delete_log_group(logGroupName=INTERCEPTOR_LOG_GROUP)
#     print(f"Deleted log group: {INTERCEPTOR_LOG_GROUP}")
# except logs_client.exceptions.ResourceNotFoundException:
#     pass
# except Exception as e:
#     print(f"Log group delete ({INTERCEPTOR_LOG_GROUP}): {e}")
#
# # ── Phase 5: Interceptor IAM role is managed by CloudFormation ──────────────
# # The role (AnyCompanyPIIInterceptorLambdaRole) is created by
# # infra/stacks/l4-security.yaml and removed with the stack. Do NOT delete it here.
# print("Interceptor IAM role is CloudFormation-managed; skipping (removed with the stack).")
#
# # ── Phase 6: Delete the Bedrock Guardrail and its SSM parameters ───────────
# try:
#     for page in bedrock.get_paginator("list_guardrails").paginate():
#         for gr in page.get("guardrails", []):
#             if gr.get("name") == GUARDRAIL_NAME:
#                 bedrock.delete_guardrail(guardrailIdentifier=gr["id"])
#                 print(f"Deleted guardrail: {GUARDRAIL_NAME} ({gr['id']})")
# except Exception as e:
#     print(f"Guardrail cleanup: {e}")
# for key in ["guardrail_id", "guardrail_version"]:
#     try:
#         ssm.delete_parameter(Name=f"{SSM_PREFIX}/{key}")
#         print(f"Deleted SSM parameter: {SSM_PREFIX}/{key}")
#     except ssm.exceptions.ParameterNotFound:
#         pass
#
# print("\n✅ Sensitive data masking resources cleaned up.")